In [172]:
import os
from dotenv import load_dotenv,find_dotenv

_ = load_dotenv(find_dotenv(),override=True)

openai_key = os.getenv("OPENAI_API_KEY")

In [173]:
from langgraph.graph import StateGraph,END
from  typing import TypedDict, Annotated
import operator
from  langchain_core.messages import AnyMessage,SystemMessage,HumanMessage,ToolMessage
from  langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()


In [174]:
from uuid import uuid4

In [175]:
def reduce_messages(left: list[AnyMessage],right: list[AnyMessage])-> list[AnyMessage]:
    for message in right:
        if not message.id:
            message.id = str(uuid4())

    merged = left.copy()
    for message in right:
        for i , existing in enumerate(merged):
            if existing.id == message.id:
                merged[i] = message
                break
        else:
            merged.append(message)
    return merged


class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage],reduce_messages]









In [176]:
tool = TavilySearchResults(max_results=2)

In [177]:
class Agent:
    def __init__(self, model, tools, checkpointer = None, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer,
                                   interrupt_before=["action"])
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [178]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(model = "openai/gpt-3.5-turbo",base_url="https://openrouter.ai/api/v1",api_key=openai_key)
#with SqliteSaver.from_conn_string(":memory:") as memory:
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [179]:
messages = [HumanMessage(content="Whats the weather in SF?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 152, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.0001075, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.0001075, 'upstream_inference_prompt_cost': 7.6e-05, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780708380-uuLO7SNt4Q8ZZnxBbyiO', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9a7d-896e-7a71-99bf-1293c89aaf12-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in San Francisco'}, 'id': 'call_UfcmTokRSbaP

In [180]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='75ad8f2d-0301-49b7-be9e-a5c5b53b9ee1'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 152, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.0001075, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.0001075, 'upstream_inference_prompt_cost': 7.6e-05, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780708380-uuLO7SNt4Q8ZZnxBbyiO', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9

In [181]:
abot.graph.get_state(thread).next

('action',)

In [182]:
for event in abot.graph.stream(None,thread):
    for v in event.values():
        print(v)

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'weather in San Francisco'}, 'id': 'call_UfcmTokRSbaPvgArsnmg5PxE', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='[{\'title\': \'San Francisco, CA Monthly Weather - AccuWeather\', \'url\': \'https://www.accuweather.com/en/us/san-francisco/94103/june-weather/347629\', \'content\': "## Temperature Graph\\n\\n°F\\n\\nAvg. Hi\\n\\nAvg. Lo\\n\\nActual Hi\\n\\nActual Lo\\n\\nForecast Hi\\n\\nForecast Lo\\n\\n## June Weather in San Francisco\\n\\n San Francisco\'s June 2026 forecast shows daily high temperatures ranging from 61° to 75°, with overnight lows between 52° and 58°. The average high for June is 67° with an average low of 55°. AccuWeather\'s monthly forecast extends further ahead than any other source, with day-by-day RealFeel® Temperatures giving a complete picture of how June weather will actually feel in San Francisco. See also: July | August | September. \\n\\n## Further Ahead\\n\\n###

In [183]:
abot.graph.get_state(thread)


StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='75ad8f2d-0301-49b7-be9e-a5c5b53b9ee1'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 152, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.0001075, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.0001075, 'upstream_inference_prompt_cost': 7.6e-05, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780708380-uuLO7SNt4Q8ZZnxBbyiO', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9

In [184]:
abot.graph.get_state(thread).next

()

In [185]:
messages = [HumanMessage("what is the weather in LA")]
thread = {"configurable":{"thread_id":"2"}}
for event in abot.graph.stream({"messages":messages},thread):
    for v in event.values():
        print(v)

while abot.graph.get_state(thread).next:
    print(abot.graph.get_state(thread))
    _input = input("Proceed????????????")

    if _input != "y":
        print("aborting")
        break

    for event in abot.graph.stream(None, thread):
        for v in event.values():
            print(v)

{'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 152, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.0001075, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.0001075, 'upstream_inference_prompt_cost': 7.6e-05, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780708387-He7nNJcpaEHtVB9bWmLa', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9a7d-a4ed-7441-bf6c-a6aa81d35e38-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in Los Angeles'}, 'id': 'call_UrNt7Tj7jX0ngw

In [186]:
messages = [HumanMessage("What is the weather in LA?")]
thread = {"configurable":{"thread_id":"3"}}
for event in abot.graph.stream({"messages":messages},thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 153, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000108, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000108, 'upstream_inference_prompt_cost': 7.65e-05, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780708399-SNeHt1KyHZ6sMahH5QW9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9a7d-d3f5-7582-b8cb-622314ce339a-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in Los Angeles'}, 'id': 'call_9YXy6L3FCEZnnlE

In [187]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='What is the weather in LA?', additional_kwargs={}, response_metadata={}, id='519742fa-2b94-4653-b728-a47d49c0abd3'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 153, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000108, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000108, 'upstream_inference_prompt_cost': 7.65e-05, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780708399-SNeHt1KyHZ6sMahH5QW9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e

In [188]:
current_values = abot.graph.get_state(thread)

In [189]:
current_values.values['messages'][-1]

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 153, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000108, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000108, 'upstream_inference_prompt_cost': 7.65e-05, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780708399-SNeHt1KyHZ6sMahH5QW9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9a7d-d3f5-7582-b8cb-622314ce339a-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in Los Angeles'}, 'id': 'call_9YXy6L3FCEZnnlEM7WkRw9cu', 't

In [190]:
current_values.values['messages'][-1].tool_calls

[{'name': 'tavily_search_results_json',
  'args': {'query': 'weather in Los Angeles'},
  'id': 'call_9YXy6L3FCEZnnlEM7WkRw9cu',
  'type': 'tool_call'}]

In [ ]:
_id = current_values.values['messages'][-1].tool_calls[0]['id']

current_values.values['messages'][-1].tool_calls = [
    {'name': 'tavily_search_results_json',
     'args': {'query': 'current weather in Louisiana'},
     'id': _id}
]



In [192]:
abot.graph.update_state(thread,current_values.values)

{'configurable': {'thread_id': '3',
  'checkpoint_ns': '',
  'checkpoint_id': '1f16144e-7266-620f-8002-a741b6489d97'}}

In [193]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='What is the weather in LA?', additional_kwargs={}, response_metadata={}, id='519742fa-2b94-4653-b728-a47d49c0abd3'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 153, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000108, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000108, 'upstream_inference_prompt_cost': 7.65e-05, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780708399-SNeHt1KyHZ6sMahH5QW9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e

In [194]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Louisiana'}, 'id': 'call_9YXy6L3FCEZnnlEM7WkRw9cu', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='[{\'title\': \'New Orleans, LA Monthly Weather | AccuWeather\', \'url\': \'https://www.accuweather.com/en/us/new-orleans/70112/june-weather/348585\', \'content\': \'## 2026\\n\\n## 10-Day\\n\\nHeavy t-storms in the p.m.\\nHumid with heavy t-storms\\nShowers and a heavier t-storm\\nHumid with a thunderstorm\\nHumid with a shower\\nMostly sunny and humid\\nVariable clouds with showers\\nRain tapering off\\nHumid with a few showers\\nHumid with showers around\\nHumid with rain at times\\nHumid with rain tapering off\\nHumid with rain\\nA t-storm around in the p.m.\\nA shower and thunderstorm\\nA p.m. t-storm possible\\nHumid with a t-storm possible\\nA t-storm around in the p.m.\\nHumid; a stray p.m. t-shower\\nMostly cloudy and humid\\nA shower and thunderstorm\\nAn afternoon t-

In [195]:
states =[]
for state in abot.graph.get_state_history(thread):
    print(state)
    print('--##################---------')
    states.append(state)

StateSnapshot(values={'messages': [HumanMessage(content='What is the weather in LA?', additional_kwargs={}, response_metadata={}, id='519742fa-2b94-4653-b728-a47d49c0abd3'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 153, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000108, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000108, 'upstream_inference_prompt_cost': 7.65e-05, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780708399-SNeHt1KyHZ6sMahH5QW9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e

In [217]:
to_replay = states[-3]

In [218]:
to_replay

StateSnapshot(values={'messages': [HumanMessage(content='What is the weather in LA?', additional_kwargs={}, response_metadata={}, id='519742fa-2b94-4653-b728-a47d49c0abd3'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 153, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000108, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000108, 'upstream_inference_prompt_cost': 7.65e-05, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780708399-SNeHt1KyHZ6sMahH5QW9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e

In [219]:
for event in abot.graph.stream(None,to_replay.config):
    for k,v in event.items():
        print(v)

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'weather in Los Angeles'}, 'id': 'call_9YXy6L3FCEZnnlEM7WkRw9cu', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='[{\'title\': \'Los Angeles weather in June 2026 | California, USA\', \'url\': \'https://www.weather2travel.com/california/los-angeles/june\', \'content\': "The June weather guide for California (Los Angeles) shows long term weather averages processed from data supplied by CRU (University of East Anglia), the Met Office & the Netherlands Meteorological Institute. Find out more about our data sources.\\n\\n### More about Los Angeles\\n\\nA sports lover\'s guide to Los Angeles\\nA sports lover\'s guide to Los Angeles\\nHow to explore LA on a budget\\nHow to explore LA on a budget\\nLos Angeles for beginners\\nLos Angeles for beginners\\n\\n### How hot is it in Los Angeles in June?\\n\\nDaytime temperatures usually reach 26°C in Los Angeles in June with low heat and humidity, falling to

In [220]:
_id = to_replay.values['messages'][-1].tool_calls[0]['id']
to_replay.values['messages'][-1].tool_calls = [{'name': 'tavily_search_results_json',
  'args': {'query': 'current weather in LA, accuweather'},
  'id': _id}]

In [221]:
branch_state = abot.graph.update_state(to_replay.config,to_replay.values)

In [222]:
for event in abot.graph.stream(None,branch_state):
    for k,v in event.items():
        print(v)

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in LA, accuweather'}, 'id': 'call_9YXy6L3FCEZnnlEM7WkRw9cu', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='[{\'title\': \'Los Angeles, CA Monthly Weather | AccuWeather\', \'url\': \'https://www.accuweather.com/en/us/los-angeles/90012/june-weather/347625\', \'content\': \'### Winter Center\\n\\n## Monthly\\n\\n## June\\n\\n## 2026\\n\\n## 10-Day\\n\\nSome clouds, then sunshine\\nMostly sunny\\nLow clouds, then sunshine\\nLow clouds, then sun\\nClouds breaking for sun\\nLow clouds, then sunshine\\nMostly sunny\\nPlenty of sunshine\\nPlenty of sunshine\\nPlenty of sunshine\\nMostly sunny\\nPlenty of sunshine\\nUninterrupted sunshine\\nMainly cloudy\\nMostly sunny\\nMostly sunny\\nTimes of clouds and sun\\nTimes of clouds and sun\\nMostly sunny\\nMore sun than clouds\\nMostly sunny\\nSunshine\\nSunshine and a few clouds\\nSun followed by clouds\\nSome sunshine\\nMostly cloudy\\nT

In [223]:
to_replay

StateSnapshot(values={'messages': [HumanMessage(content='What is the weather in LA?', additional_kwargs={}, response_metadata={}, id='519742fa-2b94-4653-b728-a47d49c0abd3'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 153, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000108, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000108, 'upstream_inference_prompt_cost': 7.65e-05, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780708399-SNeHt1KyHZ6sMahH5QW9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e

In [224]:
_id = to_replay.values['messages'][-1].tool_calls[0]['id']


In [225]:
state_update = {"messages":[ToolMessage(tool_call_id=_id,name="tavily_search_results_json",content="54 degree celcius")]}



In [228]:
branch_as_node = abot.graph.update_state(to_replay.config,state_update,as_node="action")

In [229]:
for event in abot.graph.stream(None,branch_as_node):
    for k,v in event.items():
        print(v)

{'messages': [AIMessage(content='The current weather in Los Angeles is 54 degrees Celsius.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 191, 'total_tokens': 204, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000115, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000115, 'upstream_inference_prompt_cost': 9.55e-05, 'upstream_inference_completions_cost': 1.95e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780710901-374bKQ1XCwJgGchFDTKq', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e9aa4-01d1-7851-9323-6b3c4f988e48-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19